# `itertools` & `functools` — Lazy Iteration & Functional Building Blocks

`itertools` implements fast, memory-efficient C-level iterator building blocks that process streams **lazily** ($O(1)$ extra memory instead of materializing huge intermediate lists).

`functools` provides higher-order utilities for memoization, partial application, reductions, and decorators.

In [1]:
import functools
import itertools
import operator

---
## 1. Core `itertools` Patterns

| Function | Purpose | Memory |
|---|---|---|
| `chain` / `chain.from_iterable` | Flatten / concatenate iterables lazily | $O(1)$ |
| `islice(it, start, stop, step)` | Slice any iterator/generator without converting to `list` | $O(1)$ |
| `batched(it, n)` *(Python 3.12+)* | Chunk streams into fixed-size batches (great for ML inference!) | $O(n)$ |
| `accumulate(it, func)` | Running prefix sums / products / max | $O(1)$ |
| `groupby(it, key)` | Group **consecutive** equal keys (**must sort first** if grouping globally!) | $O(1)$ |
| `product`, `permutations`, `combinations` | Combinatorial generators (hyperparameter grids, pairs, subsets) | $O(1)$ |

In [2]:
# 1. Lazy flattening and slicing
batches = [[1, 2, 3], [4, 5], [6, 7, 8, 9]]
flat_slice = list(itertools.islice(itertools.chain.from_iterable(batches), 2, 7))
print("chain + islice [2:7]:", flat_slice)

# 2. Fixed-size batching (Python 3.12+)
samples = range(10)
print("batched(3):", list(itertools.batched(samples, 3)))

# 3. Prefix sums & running max with accumulate
nums = [3, 1, 4, 1, 5, 9]
print("Prefix sums:", list(itertools.accumulate(nums)))
print("Running max:", list(itertools.accumulate(nums, max)))

# 4. groupby GOTCHA: only groups CONSECUTIVE items — sort by key first!
records = [("DE", "Berlin"), ("FR", "Paris"), ("DE", "Munich"), ("CH", "Zurich")]
records_sorted = sorted(records, key=lambda r: r[0])
grouped = {country: [city for _, city in grp] for country, grp in itertools.groupby(records_sorted, key=lambda r: r[0])}
print("groupby (after sorting):", grouped)

# 5. Hyperparameter grid with itertools.product
grid = list(itertools.product([1e-4, 3e-4], [16, 32], ["cosine", "linear"]))
print(f"Hyperparameter configs ({len(grid)} total), first 3:", grid[:3])
print("2-combinations of ['A', 'B', 'C', 'D']:", list(itertools.combinations(["A", "B", "C", "D"], 2)))

chain + islice [2:7]: [3, 4, 5, 6, 7]
batched(3): [(0, 1, 2), (3, 4, 5), (6, 7, 8), (9,)]
Prefix sums: [3, 4, 8, 9, 14, 23]
Running max: [3, 3, 4, 4, 5, 9]
groupby (after sorting): {'CH': ['Zurich'], 'DE': ['Berlin', 'Munich'], 'FR': ['Paris']}
Hyperparameter configs (8 total), first 3: [(0.0001, 16, 'cosine'), (0.0001, 16, 'linear'), (0.0001, 32, 'cosine')]
2-combinations of ['A', 'B', 'C', 'D']: [('A', 'B'), ('A', 'C'), ('A', 'D'), ('B', 'C'), ('B', 'D'), ('C', 'D')]


---
## 2. Core `functools` Patterns

- **`@functools.lru_cache(maxsize=...)`** / **`@functools.cache`** *(3.9+)*: automatic $O(1)$ memoization for pure functions with hashable arguments (classic for dynamic programming & expensive lookups).
- **`functools.partial(fn, *args, **kwargs)`**: freeze arguments to create specialized callables cleanly.
- **`functools.reduce(fn, iterable, init)`**: fold a sequence left-to-right into a single value.
- **`@functools.wraps(fn)`**: preserves `__name__`, `__doc__`, and type metadata when writing decorators.

In [3]:
# 1. Memoized DP with @functools.lru_cache
@functools.lru_cache(maxsize=None)
def fib(n: int) -> int:
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print("fib(40):", fib(40), "| cache_info:", fib.cache_info())

# 2. functools.partial for reusable specialized functions
parse_hex = functools.partial(int, base=16)
print("parse_hex('ff'):", parse_hex("ff"))

# 3. functools.reduce (e.g., product of numbers)
print("Product via reduce:", functools.reduce(operator.mul, [2, 3, 4, 5], 1))

# 4. @functools.wraps preserves metadata in custom decorators
def log_calls(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs)
    return wrapper

@log_calls
def normalize(x: float) -> float:
    """Clamp value to [0, 1]."""
    return max(0.0, min(1.0, x))

print(f"Preserved metadata -> name: {normalize.__name__!r}, doc: {normalize.__doc__!r}")

fib(40): 102334155 | cache_info: CacheInfo(hits=38, misses=41, maxsize=None, currsize=41)
parse_hex('ff'): 255
Product via reduce: 120
Preserved metadata -> name: 'normalize', doc: 'Clamp value to [0, 1].'
